# 07 Phase 1/2 예측 실험

**논문 실험 설계** — 40개 조건 × 10개 알고리즘(Phase 1) → 조건별 Best × 6 임베딩(Phase 2)

| 축 | 조건 수 | 클러스터 기준 |
|----|--------|--------------|
| ML | 5 type × 4 cluster = **20** | TS2Vec+KMeans (`ML_CLUSTER`) |
| SBC | 5 type × 4 cluster = **20** | Rule-based (`SBC_CLUSTER`) |
| **합계** | **40** | |

**Phase 1 (10알고리즘):** ARIMA, Prophet, SBA, TSB, RF, XGBoost, LSTM, Autoformer, N-HiTS, iTransformer  
**Phase 2 (6 하이브리드):** Phase1 Best + PCA / FastDTW / AE / GAF-CNN / TS2Vec / PatchTST  
**지표:** WMAPE | **검증:** 201731–201733 (3주)

### ⓪ 환경 설정 및 데이터 병합

주간 데이터에 SBC·ML 클러스터 라벨을 붙이고, Phase 1/2 실험 유틸을 로드합니다.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED, SBC_CLUSTER, ML_CLUSTER
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.phase_experiments import (
    PHASE1_MODELS, EMBEDDING_NAMES,
    run_phase1_all, summarize_phase1,
    run_phase2_all, summarize_phase2,
)

print('Torch device:', device_label())

# 주간 데이터 + 피처 + 클러스터 라벨 병합
df = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
feat_path = DATA_PROCESSED / 'df_weekly_features.parquet'
feat_df = pd.read_parquet(feat_path) if feat_path.exists() else df.copy()
sbc = pd.read_parquet(SBC_CLUSTER)
ml = pd.read_parquet(ML_CLUSTER)

for d in (df, feat_df):
    d.drop(columns=[c for c in d.columns if c in ('SBC_CLUSTER', 'ML_CLUSTER')], errors='ignore', inplace=True)

cl_sbc = sbc[['type', 'family', 'SBC_CLUSTER']]
cl_ml = ml[['type', 'family', 'ML_CLUSTER', 'embedding_method', 'clustering_method']]
df = df.merge(cl_sbc, on=['type', 'family'], how='left')
df = df.merge(cl_ml, on=['type', 'family'], how='left')
feat_df = feat_df.merge(cl_sbc, on=['type', 'family'], how='left')
feat_df = feat_df.merge(cl_ml, on=['type', 'family'], how='left')

print('시계열:', df.groupby(['type', 'family']).ngroups)
print('ML 클러스터링:', ml['embedding_method'].iloc[0], '+', ml['clustering_method'].iloc[0])
print('Phase1 알고리즘:', len(PHASE1_MODELS), '개 | Phase2 임베딩:', len(EMBEDDING_NAMES), '개')
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)

### ① Phase 1 — 40조건 × 10알고리즘

각 **type×cluster** 조건(20 SBC + 20 ML)에서 10개 알고리즘을 독립 실행하고, 조건별 평균 WMAPE로 Best를 선정합니다.

> DL 모델은 연산량 절감을 위해 epoch=40 적용. 전체 실행에 수십 분~수 시간 소요될 수 있습니다.

In [ ]:
# Phase 1: SBC 20조건 + ML 20조건 (이미 실행됐으면 캐시 로드)
p1_path = DATA_PROCESSED / 'phase1_results.parquet'
if p1_path.exists():
    phase1 = pd.read_parquet(p1_path)
    phase1_summary = pd.read_csv(DATA_PROCESSED / 'phase1_summary.csv')
    phase1_best = pd.read_csv(DATA_PROCESSED / 'phase1_best_per_condition.csv')
    print('Phase1 캐시 로드 | rows:', len(phase1))
else:
    phase1_sbc = run_phase1_all(df, feat_df, cluster_col='SBC_CLUSTER', cluster_scheme='SBC')
    phase1_ml = run_phase1_all(df, feat_df, cluster_col='ML_CLUSTER', cluster_scheme='ML')
    phase1 = pd.concat([phase1_sbc, phase1_ml], ignore_index=True)
    phase1_summary, phase1_best = summarize_phase1(phase1)
    phase1.to_parquet(p1_path, index=False)
    phase1_summary.to_csv(DATA_PROCESSED / 'phase1_summary.csv', index=False)
    phase1_best.to_csv(DATA_PROCESSED / 'phase1_best_per_condition.csv', index=False)
    print('Phase1 완료 | rows:', len(phase1))
phase1_best.sort_values(['cluster_scheme', 'type', 'cluster'])

### ② Phase 1 결과 요약

조건(type×cluster)별 10알고리즘 WMAPE 평균 — **최저 WMAPE 알고리즘**이 Phase 2의 base model이 됩니다.

In [ ]:
# Phase1 조건별 Best 알고리즘 (40개)
display(phase1_best.round(2))

# 전체 알고리즘 평균 WMAPE (참고)
print('=== Phase1 알고리즘별 전체 평균 WMAPE ===')
print(phase1.groupby(['cluster_scheme', 'model'])['wmape'].mean().unstack('cluster_scheme').round(2))

### ③ Phase 2 — 40조건 × 6 임베딩 하이브리드

Phase 1 Best 알고리즘에 6종 시계열 임베딩(PCA, FastDTW, AE, GAF-CNN, TS2Vec, PatchTST)을 결합해 조건별 WMAPE를 재측정합니다.

In [ ]:
# Phase 2: 조건별 Best × 6 임베딩 (GPU 사용 시 DL·임베딩 가속)
p2_path = DATA_PROCESSED / 'phase2_results.parquet'
if p2_path.exists():
    phase2 = pd.read_parquet(p2_path)
    phase2_summary = pd.read_csv(DATA_PROCESSED / 'phase2_summary.csv')
    phase2_best = pd.read_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv')
    print('Phase2 캐시 로드 | rows:', len(phase2))
else:
    phase2_sbc = run_phase2_all(df, feat_df, 'SBC_CLUSTER', 'SBC', phase1_best)
    phase2_ml = run_phase2_all(df, feat_df, 'ML_CLUSTER', 'ML', phase1_best)
    phase2 = pd.concat([phase2_sbc, phase2_ml], ignore_index=True)
    phase2_summary, phase2_best = summarize_phase2(phase2)
    phase2.to_parquet(p2_path, index=False)
    phase2_summary.to_csv(DATA_PROCESSED / 'phase2_summary.csv', index=False)
    phase2_best.to_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv', index=False)
    print('Phase2 완료 | rows:', len(phase2))
phase2_best.sort_values(['cluster_scheme', 'type', 'cluster'])

### ④ 최종 Best 선정 (Phase 1 vs Phase 2)

40개 조건 각각에서 Phase 1 단일 모델 vs Phase 2 하이브리드 중 WMAPE가 낮은 쪽을 최종 Best로 기록합니다.

In [ ]:
# Phase1 vs Phase2 최종 비교
final_rows = []
for row in phase1_best.itertuples(index=False):
    p1_wmape = row.best_wmape
    p2_row = phase2_best[
        (phase2_best['cluster_scheme'] == row.cluster_scheme)
        & (phase2_best['type'] == row.type)
        & (phase2_best['cluster'] == row.cluster)
    ]
    if p2_row.empty:
        continue
    p2 = p2_row.iloc[0]
    if p2['best_wmape'] < p1_wmape:
        winner = 'Phase2'
        model = p2['best_hybrid']
        wmape_val = p2['best_wmape']
    else:
        winner = 'Phase1'
        model = row.best_model
        wmape_val = p1_wmape
    final_rows.append({
        'cluster_scheme': row.cluster_scheme,
        'type': row.type,
        'cluster': row.cluster,
        'winner': winner,
        'final_model': model,
        'wmape': wmape_val,
        'phase1_best': row.best_model,
        'phase1_wmape': p1_wmape,
        'phase2_best': p2['best_hybrid'],
        'phase2_wmape': p2['best_wmape'],
    })

final_best = pd.DataFrame(final_rows)
final_best.to_csv(DATA_PROCESSED / 'final_best_per_condition.csv', index=False)
print('=== 최종 Best (40조건) ===')
display(final_best.round(2))
print('\nPhase2 승리 비율:', round((final_best['winner'] == 'Phase2').mean(), 3))

## 분석 요약

### 실험 구조
- **40조건** = SBC(5×4) + ML(5×4) | **Phase1** 10알고리즘 | **Phase2** Best×6임베딩
- 조건별 WMAPE 평균으로 Best 선정 → Phase2에서 임베딩 결합 후 재비교

### 해석 가이드
- Phase1 표: 클러스터·type마다 **이론적 권고(Smooth→ARIMA 등)가 실측 Best인지** 확인
- Phase2 표: 임베딩 결합이 Phase1 대비 WMAPE를 **얼마나 개선**하는지 확인
- `final_best_per_condition.csv`: 40조건 최종 모델 → 10장 하이브리드 프레임워크 입력

> SBC 라벨은 해석·세분화 축이며, 최적 알고리즘은 **실측 WMAPE**로 결정합니다.